In [1]:
import json
import floorplan_backend
import time

In [2]:
with open('../samples/topology.json') as topology_json:
    with open('../samples/construction.json') as construction_json:
        topology = json.load(topology_json)
        construction = json.load(construction_json)
        start = time.time()
        building = floorplan_backend.Building.from_data(topology, construction)
        print(time.time()-start)
        building.solve_joints()
        geoms = {
            uuid: result
            for face_id in building.face_ids()
            for uuid, result in building.layer_geometry(face_id).items()
        }

0.005293130874633789


In [3]:
construction

{'materials': {'0cbab760-7b22-4202-9e56-4d096260f8ad': {'name': 'OSB',
   'density': 600,
   'heat_transfer_coefficient': 0.12,
   'specific_heat_capacity': 1600,
   'dictionary': {}},
  '4ca55fd2-ce4b-4c8b-b169-8c16320cebf4': {'name': 'Fir',
   'density': 400,
   'heat_transfer_coefficient': 0.11,
   'specific_heat_capacity': 1500,
   'dictionary': {}},
  '2cb9546a-e67c-4abc-a232-4a75f5c7d0c0': {'name': 'Mineral wool',
   'density': 50,
   'heat_transfer_coefficient': 0.033,
   'specific_heat_capacity': 1030,
   'dictionary': {}},
  'f1999927-0e5a-4b9e-9f46-b8cae020db32': {'name': 'Ceramic facade tiles',
   'density': 2000,
   'heat_transfer_coefficient': 1,
   'specific_heat_capacity': 1000,
   'dictionary': {}},
  '73dd0c9b-3596-4123-b48e-73cd2a2622fb': {'name': 'Concrete',
   'density': 2200,
   'heat_transfer_coefficient': 1.5,
   'specific_heat_capacity': 840,
   'dictionary': {}}},
 'composites': {'e48e81f5-5daf-4ffd-be13-665bf6da2709': {'name': 'Studs',
   'part1': '4ca55fd2-ce

In [4]:
geoms

{'79147c02-28fa-4543-8a77-d056e6f9cd03': [{'name': 'plate',
   'uuid': '5c7b651d-6557-4643-8043-b240473314c1',
   'area': 9.435037335046285,
   'joints': [{'face_Id': 'edd3de2b-38f4-44f3-b783-e79a6af1a72e',
     'layer_idx': 2},
    {'face_Id': 'ddb9ebf9-aecf-4887-a398-3a3bfb550110', 'layer_idx': 2},
    {'face_Id': '034a76a5-be0d-4daa-8eb6-dc6f082193cd', 'layer_idx': 0},
    {'face_Id': 'c0debd01-5375-4cf4-a2df-d39f0c669fbd', 'layer_idx': 0},
    {'face_Id': 'ed46f781-79d0-4c01-98b0-d2862b16270f', 'layer_idx': 0},
    {'face_Id': 'de7fafbc-e50e-4930-b677-0df9045beb58', 'layer_idx': 0}],
   'inner': [(8.925170429216662, 7.392022311882962, 6.0),
    (7.9750613071576595, 7.629549592397557, 6.0),
    (8.004165582162, 7.7459666924149975, 6.0),
    (5.938574464185, 7.745966692414999, 6.0),
    (5.938574464185001, 5.680375574438001, 6.0),
    (5.938574464184999, 4.647580015448999, 6.0),
    (9.174665183437828, 4.647580015449004, 6.0)],
   'outer': [(8.934486321877857, 7.400001102781708, 6.01

In [5]:
import numpy as np
import plotly.graph_objects as go

display_material = 'studs'

def polygon_mesh(face):
    face = np.asarray(face)
    n = len(face)
    return (face[:, 0],face[:, 1],face[:, 2],np.zeros(n - 2, dtype=int),np.arange(1, n - 1),np.arange(2, n))

fig = go.Figure()

for geom in geoms.values():
    colour = np.random.randint(0,255,3)
    colour = f"rgb({colour[0]}, {colour[1]}, {colour[2]})"
    print(geom)
    for layer in geom:
        print(layer)
        for face in [layer['inner'],layer['outer']]:
            x, y, z, i, j, k = polygon_mesh(face)
            
            fig.add_trace(go.Mesh3d(x=x,y=y,z=z,i=i,j=j,k=k,opacity=1,flatshading=True,color=colour,name=layer["name"]))

fig.update_layout(scene=dict(aspectmode="data",xaxis_title="X",yaxis_title="Y",zaxis_title="Z",),)
fig.write_html("geometry.html", auto_open=True)
#fig.show()

[{'name': 'plate', 'uuid': '5c7b651d-6557-4643-8043-b240473314c1', 'area': 9.435037335046285, 'joints': [{'face_Id': 'edd3de2b-38f4-44f3-b783-e79a6af1a72e', 'layer_idx': 2}, {'face_Id': 'ddb9ebf9-aecf-4887-a398-3a3bfb550110', 'layer_idx': 2}, {'face_Id': '034a76a5-be0d-4daa-8eb6-dc6f082193cd', 'layer_idx': 0}, {'face_Id': 'c0debd01-5375-4cf4-a2df-d39f0c669fbd', 'layer_idx': 0}, {'face_Id': 'ed46f781-79d0-4c01-98b0-d2862b16270f', 'layer_idx': 0}, {'face_Id': 'de7fafbc-e50e-4930-b677-0df9045beb58', 'layer_idx': 0}], 'inner': [(8.925170429216662, 7.392022311882962, 6.0), (7.9750613071576595, 7.629549592397557, 6.0), (8.004165582162, 7.7459666924149975, 6.0), (5.938574464185, 7.745966692414999, 6.0), (5.938574464185001, 5.680375574438001, 6.0), (5.938574464184999, 4.647580015448999, 6.0), (9.174665183437828, 4.647580015449004, 6.0)], 'outer': [(8.934486321877857, 7.400001102781708, 6.01), (7.977486663408021, 7.639251017399011, 6.01), (8.004165582162, 7.755966692414998, 6.01), (5.9285744641